Cell 1: Environment Anchoring, Pathing & Strategic Optimization TargetsT

his initialization cell mirrors the previous dynamic pathlib root-finding setup to ensure reproducibility while declaring your core actuarial optimization benchmarks.

In [ ]:
%load_ext autoreload
%autoreload 2


import os
import sys
import sqlite3
import logging
import numpy as np
import pandas as pd
from pathlib import Path

# =====================================================================
# 1. LOGGING & DYNAMIC ANCHOR RECOVERY INFRASTRUCTURE
# =====================================================================
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("Notebook_3_Pricing_Engine")

notebook_path = Path(os.getcwd())
root_dir = notebook_path
while root_dir.name != "data_warehouse" and root_dir.parent != root_dir:
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# =====================================================================
# 2. GLOBAL STRATEGIC FUND PROPERTIES & DYNAMIC SUBSCRIPTION CONTROLLERS
# =====================================================================
TOTAL_FUND_CORPUS = 15_000_000        # $30 Million Launch Corpus
SUBSCRIPTION_RATE = 1.0               # Strict 1:1 capacity limit
SINGLE_INDUSTRY_CAP_PCT = 0.2        # 15% single-industry concentration rule
AVG_LOAN_SIZE = 250_000               # Baseline underlying loan size
FUND_MAX_FIRST_LOSS_PCT = 0.20        # Capped absolute contract ceiling (20%)

# --- THE NEW OPERATIONAL REALITY VARIABLES ---
# Enforces the average requested initial subscription size (e.g., 15% instead of 20% max = 0.75 utilization)
AVG_SUBSCRIPTION_UTILIZATION_PCT = 0.75 

# Estimated annual portfolio principal paydown rate due to standard loan amortization schedules
EST_ANNUAL_AMORTIZATION_PAYDOWN = 0.12  

# Conservative Investment Interest Yield Floor
CORPUS_INTEREST_RATE = 0.0250         

# Global Short-Duration Asset Cap (20% maximum)
SHORT_CLASS_ALLOCATION_CAP = 0.30   

# Actuarial Target Settings
TARGET_CONFIDENCE_INTERVAL = 0.95     # 95% Probability of Fund Growth required

# Proposed Baseline Rates for Evaluation Comparisons
PROPOSED_SHORT_FEE = 0.050
PROPOSED_LONG_FEE = 0.035

# --- CALCULATED INSTITUTIONAL CONSTANTS ---
# Absolute maximum contract limit per loan
CEILING_EXPOSURE_PER_LOAN = AVG_LOAN_SIZE * FUND_MAX_FIRST_LOSS_PCT  # $50,000

# THE REALISTIC UNDERWRITING BASELINE EXPOSURE SIZE
AVG_SUBSCRIPTION_EXPOSURE = CEILING_EXPOSURE_PER_LOAN * AVG_SUBSCRIPTION_UTILIZATION_PCT  # e.g., $37,500

MAX_TOTAL_FUND_EXPOSURE = TOTAL_FUND_CORPUS * SUBSCRIPTION_RATE  # $30,000,000
MAX_RISK_PER_INDUSTRY = MAX_TOTAL_FUND_EXPOSURE * SINGLE_INDUSTRY_CAP_PCT  # $4,500,000

# Notice how our maximum capacity scales up naturally based on your utilization variable
MAX_ACTIVE_SUBS_PER_INDUSTRY = int(MAX_RISK_PER_INDUSTRY // AVG_SUBSCRIPTION_EXPOSURE) 

print("--- Calibrated Underwriting Parameters Enforced ---")
print(f" • Total Core Corpus Capital     : ${TOTAL_FUND_CORPUS:,.2f}")
print(f" • Max Ceiling Guarantee Limit   : ${CEILING_EXPOSURE_PER_LOAN:,.2f}")
print(f" • EXPECTED AVG SUBSCRIPTION SIZE: ${AVG_SUBSCRIPTION_EXPOSURE:,.2f} ({AVG_SUBSCRIPTION_UTILIZATION_PCT:.0%} Utilization)")
print(f" • Estimated Portfolio Amortization: -{EST_ANNUAL_AMORTIZATION_PAYDOWN:.1%} Principal Paydown/Year")
print(f" • Single-Industry Limit (15%)   : ${MAX_RISK_PER_INDUSTRY:,.2f} Max Risk/Sector")



--- Calibrated Underwriting Parameters Enforced ---
 • Total Core Corpus Capital     : $15,000,000.00
 • Max Ceiling Guarantee Limit   : $50,000.00
 • EXPECTED AVG SUBSCRIPTION SIZE: $37,500.00 (75% Utilization)
 • Estimated Portfolio Amortization: -12.0% Principal Paydown/Year
 • Single-Industry Limit (15%)   : $3,000,000.00 Max Risk/Sector


Cell 2: Transitory Data Ingestion & Actuarial Life-Table Compilers

This cell imports micro-records from the local transitory database and compiles the identical Kaplan-Meier and conditional probability engines used in previous work.

In [53]:
# =====================================================================
# TRANSITORY WAREHOUSE DATA INGESTION
# =====================================================================
db_path = root_dir / "databases" / "transitory" / "peri_urban_ag_analysis.db"

if not db_path.exists():
    raise FileNotFoundError(
        f"Missing mandatory data asset at verified path: {db_path}\n"
        f"Please run Notebook 1 serialization cells first to generate this transitory layer."
    )

conn = sqlite3.connect(db_path)
df_loans = pd.read_sql_query("SELECT * FROM source_loans_snapshot", conn)
conn.close()

WINDOW_MONTHS = 60
ANNUAL_CHECKPOINTS = np.arange(12, WINDOW_MONTHS + 1, 12)

print(f"🎉 Connection Established via Pathlib.")
print(f" • Target Database: {db_path.name}")
print(f" • Micro-Records Ingested to RAM: {len(df_loans):,}")



🎉 Connection Established via Pathlib.
 • Target Database: peri_urban_ag_analysis.db
 • Micro-Records Ingested to RAM: 91,169


Cell 3: Portfolio Extraction Matrix Mapping (Routing Strategy Implementation)

This maps the 25 target NAICS codes, dynamically routing sparse cells (N < 20) to their 3-digit parent neighborhood proxies to protect against high-variance data gaps.

In [54]:
# =====================================================================
# LIFE-TABLE MATHEMATICAL & DATA ROUTING ENGINES
# =====================================================================
def extract_actuarial_survival_vector(group_df):
    if len(group_df) == 0:
        return np.zeros(len(ANNUAL_CHECKPOINTS))
        
    monthly_stats = group_df.groupby('survival_months').agg(
        d_i=('event_occurred', 'sum'),
        total_exits=('survival_months', 'count')
    ).sort_index()
    
    total_records = len(group_df)
    cumulative_exits_prior = monthly_stats['total_exits'].cumsum().shift(1).fillna(0)
    monthly_stats['n_i'] = total_records - cumulative_exits_prior
    
    monthly_stats['step_survival'] = np.where(
        monthly_stats['n_i'] > 0,
        1.0 - (monthly_stats['d_i'] / monthly_stats['n_i']),
        1.0
    )
    monthly_stats['cumulative_survival'] = monthly_stats['step_survival'].cumprod()
    
    cumulative_defaults = []
    for month in ANNUAL_CHECKPOINTS:
        historical_steps = monthly_stats[monthly_stats.index <= month]
        latest_survival = historical_steps['cumulative_survival'].iloc[-1] if len(historical_steps) > 0 else 1.0
        cumulative_defaults.append(1.0 - latest_survival)
        
    return np.array(cumulative_defaults)

def calculate_peak_marginal_pd(cumulative_defaults):
    F = np.insert(cumulative_defaults, 0, 0.0)
    marginal_pds = []
    for t in range(1, len(F)):
        s_prev = 1.0 - F[t-1]
        q_t = (F[t] - F[t-1]) / s_prev if s_prev > 0 else 0.0
        marginal_pds.append(q_t)
    return max(marginal_pds) if len(marginal_pds) > 0 else 0.0

# --- ROUTING MATRIX ENGINE (SPARSE CONTROLLER INTERCEPTOR) ---
SPARSE_STRATEGY = 'PARENT_HIERARCHY'
pricing_registry = []
df_targets_only = df_loans[df_loans['is_core_sample'] == 1]
unique_naics_targets = df_targets_only['naics_4d'].unique()

for naics in unique_naics_targets:
    for is_long in [0, 1]:
        native_cell = df_targets_only[(df_targets_only['naics_4d'] == naics) & (df_targets_only['is_long_duration'] == is_long)]
        is_sparse = len(native_cell) < 20 or (native_cell['is_sparse_cell'].iloc[0] == 1 if len(native_cell) > 0 else True)
        
        routing_strategy = "NATIVE_CELL"
        active_data = native_cell
        
        if is_sparse:
            if SPARSE_STRATEGY == 'PARENT_HIERARCHY':
                parent_3d = naics[:3]
                active_data = df_loans[(df_loans['naics_3d'] == parent_3d) & (df_loans['is_long_duration'] == is_long)]
                routing_strategy = "PARENT_HIERARCHY" if len(active_data) >= 20 else "PENALTY_BOX_FLOOR"
            else:
                routing_strategy = "PENALTY_BOX_FLOOR"
                active_data = pd.DataFrame()

        if routing_strategy == "PENALTY_BOX_FLOOR" or len(active_data) == 0:
            peak_empirical_q = 0.065
            sample_size = len(native_cell)
        else:
            cum_curve = extract_actuarial_survival_vector(active_data)
            peak_empirical_q = calculate_peak_marginal_pd(cum_curve)
            sample_size = len(active_data)
            
        pricing_registry.append({
            'naics_4d': naics, 'is_long_duration': is_long,
            'routing_strategy': routing_strategy, 'sample_size_used': sample_size,
            'peak_annual_pd': peak_empirical_q
        })

df_pricing_matrix = pd.DataFrame(pricing_registry)
print(f"🎉 Target risk profile grid successfully compiled over {len(df_pricing_matrix)} active cells.")


🎉 Target risk profile grid successfully compiled over 50 active cells.


Cell 4: Independent Asset Class Optimization Engine

This is the core new asset. It processes two completely decoupled Monte Carlo simulation tracks—one assuming the pool is 100% short-duration working capital loans, and the other assuming it is 100% long-duration equipment/real estate debt. It then sweeps a tight pricing grid to resolve the precise 95% breakeven markers.

In [56]:
# =====================================================================
# CELL 4: BLENDED PORTFOLIO MONTE CARLO OPTIMIZATION (AMORTIZATION-AWARE)
# =====================================================================
def execute_amortization_aware_optimization(df_pricing, n_simulations=10000):
    np.random.seed(42)
    available_naics = df_pricing['naics_4d'].unique()
    
    sim_pipeline_buffer = 2500
    pipeline_naics = np.random.choice(available_naics, size=sim_pipeline_buffer)
    
    industry_active_counts = {naics: 0 for naics in available_naics}
    accepted_portfolio = []
    
    short_exposure_running = 0
    total_exposure_running = 0
    
    # 1. Build the capacity-constrained blended portfolio using the new average metrics
    for naics in pipeline_naics:
        if total_exposure_running >= MAX_TOTAL_FUND_EXPOSURE:
            break
            
        is_long = np.random.choice([0,1], p=[0.40, 0.60])
        if is_long == 0 and short_exposure_running >= (MAX_TOTAL_FUND_EXPOSURE * SHORT_CLASS_ALLOCATION_CAP):
            is_long = 1  
            
        if industry_active_counts[naics] < MAX_ACTIVE_SUBS_PER_INDUSTRY:
            match = df_pricing[(df_pricing['naics_4d'] == naics) & (df_pricing['is_long_duration'] == is_long)]
            base_pd = match['peak_annual_pd'].values if len(match) > 0 else 0.04
            
            industry_active_counts[naics] += 1
            total_exposure_running += AVG_SUBSCRIPTION_EXPOSURE
            if is_long == 0:
                short_exposure_running += AVG_SUBSCRIPTION_EXPOSURE
                
            accepted_portfolio.append({
                'is_long_duration': is_long,
                'baseline_annual_pd': base_pd
            })
            
    df_active_fund = pd.DataFrame(accepted_portfolio)
    n_short = len(df_active_fund[df_active_fund['is_long_duration'] == 0])
    n_long = len(df_active_fund[df_active_fund['is_long_duration'] == 1])
    
    # 2. Extract baseline hazard rates
    pds_baseline = np.clip(df_active_fund['baseline_annual_pd'].to_numpy(), 0.0, 0.95)
    
    # 3. Process Amortization Paydowns across simulations
    # We simulate which year a loan fails in based on uniform distribution across our 5-year timeline
    random_matrix = np.random.rand(n_simulations, len(df_active_fund))
    default_matrix = random_matrix < pds_baseline
    
    # Calculate unique paydown multipliers based on default year positioning
    # Earlier years face higher exposure, later years face smaller paid-down balances
    sim_default_years = np.random.randint(1, 6, size=(n_simulations, len(df_active_fund)))
    paydown_multipliers = 1.0 - ((sim_default_years - 1) * EST_ANNUAL_AMORTIZATION_PAYDOWN)
    paydown_multipliers = np.clip(paydown_multipliers, 0.20, 1.0) # Ensure a floor protection level
    
    # Multiply the true default matrix by the dynamic paid-down exposure balances
    simulated_claims = (default_matrix * paydown_multipliers).sum(axis=1) * AVG_SUBSCRIPTION_EXPOSURE
    
    # --- REVENUE INFLOW ENGINE ---
    guaranteed_interest_income = TOTAL_FUND_CORPUS * CORPUS_INTEREST_RATE
    
    # --- OPTIMIZATION SEARCH GRID ---
    fee_scan_grid = np.linspace(0.001, 0.15, 300)
    optimal_short_fee, optimal_long_fee = None, None
    success_probability = 0
    
    for long_fee in fee_scan_grid:
        short_fee = long_fee + 0.015
        
        # Premiums collected strictly on the realistic, expected average subscription size ($37,500)
        premium_revenue_short = n_short * AVG_SUBSCRIPTION_EXPOSURE * short_fee
        premium_revenue_long = n_long * AVG_SUBSCRIPTION_EXPOSURE * long_fee
        total_premium_collected = premium_revenue_short + premium_revenue_long
        
        total_fund_revenue_inflow = total_premium_collected + guaranteed_interest_income
        prob_growth = np.sum((total_fund_revenue_inflow - simulated_claims) >= 0) / n_simulations
        
        if prob_growth >= TARGET_CONFIDENCE_INTERVAL:
            optimal_short_fee = short_fee
            optimal_long_fee = long_fee
            success_probability = prob_growth
            break

    # =====================================================================
    # AMORTIZATION-AWARE PRESENTATION PRINT MATRIX
    # =====================================================================
    print("\n" + "="*65)
    print("      PERI-URBAN FUND: AMORTIZATION-AWARE PRICING MATRIX")
    print("="*65)
    print(f" INITIAL CAPITAL ALLOCATION POOL   : ${TOTAL_FUND_CORPUS:,.2f}")
    print(f" CORPUS CONSERVATIVE INTEREST FLOOR : {CORPUS_INTEREST_RATE:.2%} (${guaranteed_interest_income:,.2f})")
    print(f" EXPECTED AVERAGE SUBSCRIPTION VALUE: ${AVG_SUBSCRIPTION_EXPOSURE:,.2f} ({AVG_SUBSCRIPTION_UTILIZATION_PCT:.0%} Avg Draw)")
    print(f" -------------------------------------------------------------")
    print(f" GLOBAL SHORT-DURATION POOL CAP    : {SHORT_CLASS_ALLOCATION_CAP:.0%} Max Face Value Allocation")
    print(f" TOTAL DEPLOYED PORTFOLIO STRUCTURE: {len(df_active_fund)} Active Guarantee Notes")
    print(f"  --> Short-Duration Working Cap   : {n_short} loans (${n_short * AVG_SUBSCRIPTION_EXPOSURE:,.2f})")
    print(f"  --> Long-Duration Infrastructure : {n_long} loans (${n_long * AVG_SUBSCRIPTION_EXPOSURE:,.2f})")
    print(f" -------------------------------------------------------------")
    
    if optimal_short_fee:
        print(f" 🟩 OPTIMIZED SHORT-DURATION FEE : {optimal_short_fee:.2%} (Your Proposed: {PROPOSED_SHORT_FEE:.2%})")
        print(f" 🟦 OPTIMIZED LONG-DURATION FEE  : {optimal_long_fee:.2%} (Your Proposed: {PROPOSED_LONG_FEE:.2%})")
        print(f"  --> Calculated Growth Safety Window: {success_probability:.2%}")
        print(f"  --> Total Collected Premium Cash   : ${total_premium_collected:,.2f}")
        print(f"  --> Average Annual Claims Paid     : ${simulated_claims.mean():,.2f}")
        print(f"  --> Projected Net Year-1 Cash Flow : ${total_fund_revenue_inflow - simulated_claims.mean():,.2f}")
    else:
        print(" ❌ ALLOCATION SYSTEM ERROR: Portfolio remains unresolvable within parameters.")
    print("="*65)

execute_amortization_aware_optimization(df_pricing_matrix)



      PERI-URBAN FUND: AMORTIZATION-AWARE PRICING MATRIX
 INITIAL CAPITAL ALLOCATION POOL   : $15,000,000.00
 CORPUS CONSERVATIVE INTEREST FLOOR : 2.50% ($375,000.00)
 EXPECTED AVERAGE SUBSCRIPTION VALUE: $37,500.00 (75% Avg Draw)
 -------------------------------------------------------------
 GLOBAL SHORT-DURATION POOL CAP    : 30% Max Face Value Allocation
 TOTAL DEPLOYED PORTFOLIO STRUCTURE: 400 Active Guarantee Notes
  --> Short-Duration Working Cap   : 120 loans ($4,500,000.00)
  --> Long-Duration Infrastructure : 280 loans ($10,500,000.00)
 -------------------------------------------------------------
 🟩 OPTIMIZED SHORT-DURATION FEE : 6.08% (Your Proposed: 5.00%)
 🟦 OPTIMIZED LONG-DURATION FEE  : 4.58% (Your Proposed: 3.50%)
  --> Calculated Growth Safety Window: 95.54%
  --> Total Collected Premium Cash   : $755,242.47
  --> Average Annual Claims Paid     : $879,201.45
  --> Projected Net Year-1 Cash Flow : $251,041.02
